# Implementasi Webinar Pertemuan 3: Career and Competition in the Industrial Era with GIS
## Studi Kasus: Analisis Aksesibilitas Fasilitas Umum
**Platform:** Python di Google Colab | **Tingkat:** Menengah

## 1. Pendahuluan
GIS (*Geographic Information System*) membantu mengelola, menganalisis, dan menyajikan data yang memiliki informasi lokasi. Notebook ini menganalisis aksesibilitas titik permukiman simulasi terhadap sekolah, puskesmas, dan rumah sakit. Jarak ke fasilitas terdekat dihitung, skor aksesibilitas dibuat berdasarkan ambang jarak, lalu hasilnya divisualisasikan dalam grafik dan peta interaktif.

**Penting:** seluruh titik dan nilai dalam notebook ini merupakan data sintetis untuk latihan, bukan lokasi fasilitas nyata atau rekomendasi resmi. Bobot dan ambang jarak adalah asumsi demonstrasi.

## 2. Rumusan Masalah, Tujuan, dan Metode
**Rumusan masalah:** Area permukiman simulasi mana yang relatif mudah atau sulit menjangkau fasilitas umum terdekat?

**Tujuan:** (1) membentuk dataset spasial, (2) menghitung jarak ke fasilitas terdekat, (3) menyusun skor aksesibilitas, (4) memvisualisasikan hasil, dan (5) mengevaluasi keterbatasan.

Jarak dihitung dengan rumus *Haversine* (kilometer). Ambang jarak ilustratif: sekolah 3 km, puskesmas 5 km, rumah sakit 10 km. Skor kategori = `max(0, 100 × (1 − jarak/ambang))`. Bobot total: sekolah 40%, puskesmas 35%, rumah sakit 25%. Skor bukan standar resmi; jarak yang dihitung adalah garis lurus, bukan jarak jalan.

## 3. Persiapan Library
Jalankan sel berikut secara berurutan. Folium digunakan untuk peta interaktif.

In [ ]:
%pip -q install folium

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
from IPython.display import display

pd.set_option("display.max_columns", None)
print("Library siap digunakan.")

## 4. Membuat Dataset Simulasi
Data dibuat di notebook, tanpa perlu mengunduh berkas. Koordinat berada pada area ilustratif dan tidak menunjukkan lokasi fasilitas sebenarnya.

In [ ]:
rng = np.random.default_rng(42)
CENTER_LAT, CENTER_LON = -3.80, 102.27
n = 30
residential = pd.DataFrame({
    "area_id": [f"AREA-{i:02d}" for i in range(1, n + 1)],
    "area_name": [f"Permukiman {i:02d}" for i in range(1, n + 1)],
    "latitude": CENTER_LAT + rng.uniform(-0.045, 0.045, n),
    "longitude": CENTER_LON + rng.uniform(-0.050, 0.050, n),
    "population": rng.integers(500, 5000, n)
})
facility_rows = [
    ("SCH-01", "Sekolah A", "Sekolah", -3.785, 102.255),
    ("SCH-02", "Sekolah B", "Sekolah", -3.815, 102.290),
    ("SCH-03", "Sekolah C", "Sekolah", -3.770, 102.305),
    ("SCH-04", "Sekolah D", "Sekolah", -3.830, 102.245),
    ("SCH-05", "Sekolah E", "Sekolah", -3.805, 102.225),
    ("SCH-06", "Sekolah F", "Sekolah", -3.840, 102.315),
    ("PKM-01", "Puskesmas A", "Puskesmas", -3.790, 102.280),
    ("PKM-02", "Puskesmas B", "Puskesmas", -3.825, 102.265),
    ("PKM-03", "Puskesmas C", "Puskesmas", -3.765, 102.250),
    ("PKM-04", "Puskesmas D", "Puskesmas", -3.845, 102.285),
    ("PKM-05", "Puskesmas E", "Puskesmas", -3.805, 102.320),
    ("RS-01", "Rumah Sakit A", "Rumah Sakit", -3.800, 102.275),
    ("RS-02", "Rumah Sakit B", "Rumah Sakit", -3.835, 102.305),
    ("RS-03", "Rumah Sakit C", "Rumah Sakit", -3.770, 102.235),
    ("RS-04", "Rumah Sakit D", "Rumah Sakit", -3.815, 102.235),
]
facilities = pd.DataFrame(facility_rows, columns=["facility_id", "facility_name", "facility_type", "latitude", "longitude"])
print(f"Titik permukiman: {len(residential)} | Fasilitas: {len(facilities)}")
display(facilities.groupby("facility_type").size().rename("jumlah").to_frame())
display(residential.head())

### 4.1 Pemeriksaan kualitas data
Memeriksa nilai kosong, ID duplikat, dan validitas lintang-bujur sebelum analisis.

In [ ]:
quality = pd.DataFrame({"pemeriksaan": [
    "Nilai kosong permukiman", "Nilai kosong fasilitas", "ID area duplikat",
    "ID fasilitas duplikat", "Lintang area di luar rentang", "Lintang fasilitas di luar rentang"
], "jumlah": [
    int(residential.isna().sum().sum()), int(facilities.isna().sum().sum()),
    int(residential.area_id.duplicated().sum()), int(facilities.facility_id.duplicated().sum()),
    int((~residential.latitude.between(-90, 90)).sum()), int((~facilities.latitude.between(-90, 90)).sum())
]})
display(quality)
assert residential.area_id.is_unique and facilities.facility_id.is_unique
assert residential[["latitude", "longitude", "population"]].notna().all().all()
assert facilities[["latitude", "longitude"]].notna().all().all()
print("Pemeriksaan dasar lulus.")

## 5. Eksplorasi Sebaran
Grafik ini memperlihatkan lokasi relatif permukiman dan fasilitas. Ini belum mengukur akses melalui jaringan jalan.

In [ ]:
plt.figure(figsize=(9, 7))
plt.scatter(residential.longitude, residential.latitude, s=30, alpha=.7, label="Permukiman")
for typ, group in facilities.groupby("facility_type"):
    plt.scatter(group.longitude, group.latitude, s=95, marker="^", label=typ)
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.title("Sebaran Simulasi Permukiman dan Fasilitas Umum")
plt.grid(alpha=.25); plt.legend(); plt.show()

## 6. Menghitung Jarak dengan Haversine
Haversine memperkirakan jarak permukaan bumi dari dua koordinat lintang-bujur. Hasilnya dalam kilometer. Untuk penerapan nyata, jarak perjalanan di jalan dan waktu tempuh dapat berbeda.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Jarak permukaan bumi antara koordinat (km)."""
    radius = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    a = np.clip(a, 0, 1)
    return 2 * radius * np.arcsin(np.sqrt(a))

rows = []
for _, area in residential.iterrows():
    for _, f in facilities.iterrows():
        rows.append({"area_id": area.area_id, "facility_id": f.facility_id,
                     "facility_type": f.facility_type,
                     "distance_km": haversine_km(area.latitude, area.longitude, f.latitude, f.longitude)})
distances = pd.DataFrame(rows)
print(f"Pasangan jarak yang dihitung: {len(distances)}")
display(distances.head())

## 7. Menentukan Fasilitas Terdekat
Untuk setiap area dan kategori fasilitas, pilih baris dengan jarak terkecil. Nama/ID fasilitas juga disimpan agar hasil dapat ditelusuri.

In [ ]:
nearest_rows = []
for (area_id, typ), group in distances.groupby(["area_id", "facility_type"]):
    row = group.loc[group.distance_km.idxmin()]
    nearest_rows.append({"area_id": area_id, "facility_type": typ,
                         "nearest_facility_id": row.facility_id, "distance_km": row.distance_km})
nearest = pd.DataFrame(nearest_rows)
distance_wide = nearest.pivot(index="area_id", columns="facility_type", values="distance_km").rename(columns={
    "Sekolah": "jarak_sekolah_km", "Puskesmas": "jarak_puskesmas_km", "Rumah Sakit": "jarak_rumah_sakit_km"})
name_wide = nearest.pivot(index="area_id", columns="facility_type", values="nearest_facility_id").rename(columns={
    "Sekolah": "sekolah_terdekat", "Puskesmas": "puskesmas_terdekat", "Rumah Sakit": "rumah_sakit_terdekat"})
results = residential.set_index("area_id").join(distance_wide).join(name_wide).reset_index()
display(results.head(10).round(3))

## 8. Membuat Skor Aksesibilitas
Skor per kategori = `max(0, 100 × (1 − jarak/ambang))`. Skor yang lebih tinggi berarti lebih dekat terhadap fasilitas menurut ambang ilustratif. Skor total menggunakan bobot sekolah 40%, puskesmas 35%, dan rumah sakit 25%.

In [ ]:
thresholds = {"jarak_sekolah_km": 3.0, "jarak_puskesmas_km": 5.0, "jarak_rumah_sakit_km": 10.0}
weights = {"sekolah": 0.40, "puskesmas": 0.35, "rumah_sakit": 0.25}
def score(distance, threshold):
    return np.maximum(0, 100 * (1 - distance / threshold))
results["skor_sekolah"] = score(results.jarak_sekolah_km, thresholds["jarak_sekolah_km"])
results["skor_puskesmas"] = score(results.jarak_puskesmas_km, thresholds["jarak_puskesmas_km"])
results["skor_rumah_sakit"] = score(results.jarak_rumah_sakit_km, thresholds["jarak_rumah_sakit_km"])
results["skor_aksesibilitas"] = (results.skor_sekolah * weights["sekolah"] +
    results.skor_puskesmas * weights["puskesmas"] + results.skor_rumah_sakit * weights["rumah_sakit"])
q1, q2 = results.skor_aksesibilitas.quantile([1/3, 2/3])
results["kategori_akses"] = np.select(
    [results.skor_aksesibilitas <= q1, results.skor_aksesibilitas <= q2],
    ["Relatif rendah", "Sedang"], default="Relatif tinggi")
results = results.sort_values("skor_aksesibilitas", ascending=False).reset_index(drop=True)
results["peringkat"] = np.arange(1, len(results) + 1)
display(results[["peringkat", "area_name", "population", "jarak_sekolah_km", "jarak_puskesmas_km",
                "jarak_rumah_sakit_km", "skor_aksesibilitas", "kategori_akses"]].head(10).round(2))

## 9. Evaluasi dan Interpretasi
Tinjau statistik jarak, skor rata-rata, area dengan skor tertinggi/terendah, dan jumlah area di atas ambang jarak. Kategori rendah/sedang/tinggi dibagi berdasarkan sebaran skor dataset simulasi, bukan standar eksternal.

In [ ]:
summary = pd.DataFrame({"Metrik": ["Jumlah permukiman", "Rata-rata jarak ke sekolah (km)",
    "Rata-rata jarak ke puskesmas (km)", "Rata-rata jarak ke rumah sakit (km)",
    "Rata-rata skor aksesibilitas", "Skor minimum", "Skor maksimum"],
    "Nilai": [len(results), results.jarak_sekolah_km.mean(), results.jarak_puskesmas_km.mean(),
    results.jarak_rumah_sakit_km.mean(), results.skor_aksesibilitas.mean(),
    results.skor_aksesibilitas.min(), results.skor_aksesibilitas.max()]})
display(summary.round(2))
print("Lima area dengan skor tertinggi")
display(results.nlargest(5, "skor_aksesibilitas")[["area_name", "skor_aksesibilitas", "kategori_akses"]].round(2))
print("Lima area dengan skor terendah")
display(results.nsmallest(5, "skor_aksesibilitas")[["area_name", "skor_aksesibilitas", "kategori_akses",
    "jarak_sekolah_km", "jarak_puskesmas_km", "jarak_rumah_sakit_km"]].round(2))
coverage = pd.DataFrame({"Kategori": ["Sekolah", "Puskesmas", "Rumah Sakit"],
    "Ambang_km": [3, 5, 10], "Area_di_atas_ambang": [
    int((results.jarak_sekolah_km > 3).sum()), int((results.jarak_puskesmas_km > 5).sum()),
    int((results.jarak_rumah_sakit_km > 10).sum())]})
coverage["Persentase_area_%"] = coverage.Area_di_atas_ambang / len(results) * 100
display(coverage.round(2))

In [ ]:
plot_data = results.sort_values("skor_aksesibilitas")
plt.figure(figsize=(10, 9))
plt.barh(plot_data.area_name, plot_data.skor_aksesibilitas)
plt.xlabel("Skor aksesibilitas (0–100)"); plt.ylabel("Titik permukiman")
plt.title("Peringkat Skor Aksesibilitas Fasilitas Umum"); plt.grid(axis="x", alpha=.25)
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 5))
plt.boxplot([results[c] for c in ["jarak_sekolah_km", "jarak_puskesmas_km", "jarak_rumah_sakit_km"]],
            labels=["Sekolah", "Puskesmas", "Rumah Sakit"])
plt.ylabel("Jarak ke fasilitas terdekat (km)"); plt.title("Distribusi Jarak")
plt.grid(axis="y", alpha=.25); plt.show()

## 10. Peta Interaktif
Warna permukiman menandai kategori akses relatif. Marker fasilitas dibedakan berdasarkan jenisnya. Klik titik untuk melihat rincian. Peta dasar OpenStreetMap memerlukan koneksi internet di browser untuk memuat tile peta; perhitungan dan dataset notebook tetap dapat dijalankan tanpa mengambil data fasilitas dari internet.

In [ ]:
m = folium.Map(location=[residential.latitude.mean(), residential.longitude.mean()], zoom_start=12)
colors = {"Relatif tinggi": "green", "Sedang": "orange", "Relatif rendah": "red"}
res_layer = folium.FeatureGroup(name="Permukiman (kategori akses)")
for _, r in results.iterrows():
    popup = (f"<b>{r.area_name}</b><br>Populasi simulasi: {int(r.population)}<br>"
             f"Skor: {r.skor_aksesibilitas:.1f}<br>Kategori: {r.kategori_akses}<br>"
             f"Sekolah: {r.jarak_sekolah_km:.2f} km<br>Puskesmas: {r.jarak_puskesmas_km:.2f} km<br>"
             f"Rumah sakit: {r.jarak_rumah_sakit_km:.2f} km")
    folium.CircleMarker([r.latitude, r.longitude], radius=5 + min(r.population / 1500, 3),
        color=colors[r.kategori_akses], fill=True, fill_opacity=.75,
        popup=folium.Popup(popup, max_width=300), tooltip=r.area_name).add_to(res_layer)
res_layer.add_to(m)
facility_colors = {"Sekolah": "blue", "Puskesmas": "purple", "Rumah Sakit": "darkred"}
for typ, group in facilities.groupby("facility_type"):
    layer = folium.FeatureGroup(name=typ)
    for _, f in group.iterrows():
        folium.Marker([f.latitude, f.longitude], tooltip=f.facility_name,
            popup=f"{f.facility_name} — data simulasi",
            icon=folium.Icon(color=facility_colors[typ], icon="info-sign")).add_to(layer)
    layer.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m

## 11. Analisis Sensitivitas Bobot
Bandingkan bobot awal dengan skenario yang memberi bobot lebih besar pada rumah sakit. Perubahan peringkat menunjukkan bahwa hasil dapat dipengaruhi oleh prioritas yang ditetapkan.

In [ ]:
sensitivity = results[["area_id", "area_name", "skor_sekolah", "skor_puskesmas", "skor_rumah_sakit"]].copy()
sensitivity["skor_A"] = sensitivity.skor_sekolah*.40 + sensitivity.skor_puskesmas*.35 + sensitivity.skor_rumah_sakit*.25
sensitivity["skor_B"] = sensitivity.skor_sekolah*.25 + sensitivity.skor_puskesmas*.35 + sensitivity.skor_rumah_sakit*.40
sensitivity["peringkat_A"] = sensitivity.skor_A.rank(ascending=False, method="min").astype(int)
sensitivity["peringkat_B"] = sensitivity.skor_B.rank(ascending=False, method="min").astype(int)
sensitivity["perubahan_peringkat"] = sensitivity.peringkat_A - sensitivity.peringkat_B
display(sensitivity.reindex(sensitivity.perubahan_peringkat.abs().sort_values(ascending=False).index)[
    ["area_name", "skor_A", "skor_B", "peringkat_A", "peringkat_B", "perubahan_peringkat"]].head(10).round(2))
print("Korelasi peringkat Spearman:", round(sensitivity.peringkat_A.corr(sensitivity.peringkat_B, method="spearman"), 3))

## 12. Kesimpulan Otomatis
Ringkasan berikut dihasilkan dari hasil perhitungan, bukan nilai yang ditulis secara manual.

In [ ]:
best = results.loc[results.skor_aksesibilitas.idxmax()]
worst = results.loc[results.skor_aksesibilitas.idxmin()]
print("RINGKASAN HASIL")
print(f"Skor tertinggi: {best.area_name} ({best.skor_aksesibilitas:.2f}).")
print(f"Skor terendah: {worst.area_name} ({worst.skor_aksesibilitas:.2f}).")
print(f"Rata-rata skor: {results.skor_aksesibilitas.mean():.2f} dari {len(results)} area.")
print("Hasil hanya berlaku untuk dataset sintetis, ambang, dan bobot pada notebook ini.")

## 13. Keterbatasan dan Pengembangan
1. Data sintetis, bukan data survei atau data resmi.
2. Jarak garis lurus tidak mempertimbangkan jaringan jalan, sungai, hambatan, kemacetan, atau moda transportasi.
3. Kapasitas, jam buka, kualitas layanan, dan jenis layanan fasilitas belum dianalisis.
4. Bobot dan ambang jarak bersifat ilustratif dan perlu divalidasi untuk studi nyata.
5. Populasi ditampilkan sebagai konteks, tetapi belum digunakan sebagai ukuran kebutuhan layanan.

Pengembangan lanjutan dapat memakai data terbuka tervalidasi, jaringan jalan dan waktu tempuh, batas administrasi, serta analisis jumlah penduduk yang terjangkau fasilitas.

## 14. Hubungan dengan Materi Webinar
Implementasi ini menunjukkan bahwa keahlian Informatika seperti Python, pengolahan data, perhitungan spasial, visualisasi, dan interpretasi dapat digabungkan untuk membantu pengambilan keputusan. GIS bukan hanya membuat peta, tetapi menghubungkan data dan kebutuhan pengguna dengan analisis yang dapat dijelaskan.

## 15. Kesimpulan
Notebook ini memperlihatkan alur analisis aksesibilitas fasilitas umum dari pembuatan data, perhitungan jarak, penyusunan skor, evaluasi, hingga visualisasi peta. Sebelum diterapkan di dunia nyata, data harus diverifikasi, asumsi dibahas bersama pemangku kepentingan, dan hasil dibandingkan dengan kondisi lapangan.